In [1]:
import os
import json
import time
from datetime import datetime

from apollo_client import OpenAI, ApolloConfig  # pattern usato negli esempi interni 
# === Apollo auth: prendi da variabili di sistema (come hai già) ===
client_id = os.getenv("APOLLO_CLIENT_ID", "")
client_secret = os.getenv("APOLLO_CLIENT_SECRET", "")
cfg = ApolloConfig(
    client_id=client_id,
    client_secret=client_secret,
    # token_url=token_url,
    # base_url=base_url
)
client = OpenAI(config=cfg)  # OpenAI-like interface usata internamente 

c:\Program Files\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_models = [m.id for m in client.models.list()]
MODELS = ["claude_3_5_haiku", "gpt-4o-mini"] 

In [21]:
with open("system_prompt.txt",'r') as sp:
    SYSTEM_MESSAGE = sp.read()
    #system_prompt = ' '.join(system_prompt.split())
with open("prompt_tipo.json", "r", encoding="utf-8") as f:
        prompts = json.load(f)

In [22]:
print(SYSTEM_MESSAGE)
print(prompts)

You are a knowledgeable assistant. Answer the user's question thoroughly and accurately.
All the questions will be related to veterinary products, as the users will ask for some help in deciding the right product to buy.
In your response try to always include a final decision on a specific brand, like brand X is preferred for those reasons.

Always respond in the same language as the user's message.

You must structure your response as a valid JSON object with exactly these fields:

{
  "answer": "your full response here",
  "decision": "your final recommendation in one sentence",
  "preferred_brand":"your preferred brand",
  "confidence": <integer from 0 to 100>,
  "confidence_rationale": "brief explanation of why you assigned this confidence score",
  "sources": ["type or name of source 1", "type or name of source 2"]
}

Rules:
- "answer" contains your complete, detailed response
- "decision" is a single clear recommendation
- "confidence" reflects how certain you are based on availa

In [26]:
import re
def clean_and_parse_json(raw_string: str) -> dict:
    """
    Pulisce una stringa JSON sporca e la parsa in un dizionario Python.
    Gestisce escape sequences problematiche, newline letterali e altri artefatti.
    """
    # 1. Rimuovi eventuali whitespace iniziali/finali
    cleaned = raw_string.strip()
    
    # 2. Prima prova il parsing diretto (caso ottimistico)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 3. Sostituisci newline LETTERALI (\n come due caratteri) nei valori stringa
    #    preservando la struttura JSON
    def fix_newlines_in_strings(s: str) -> str:
        """Converte newline letterali dentro le stringhe JSON in \\n escaped."""
        result = []
        in_string = False
        escape_next = False
        
        for i, char in enumerate(s):
            if escape_next:
                result.append(char)
                escape_next = False
                continue
            
            if char == '\\':
                escape_next = True
                result.append(char)
                continue
            
            if char == '"':
                in_string = not in_string
                result.append(char)
                continue
            
            # Se siamo dentro una stringa e troviamo un newline reale, escapalo
            if in_string and char == '\n':
                result.append('\\n')
                continue
            
            result.append(char)
        
        return ''.join(result)

    cleaned = fix_newlines_in_strings(cleaned)

    # 4. Seconda prova dopo il fix dei newline
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 5. Fallback: rimuovi caratteri di controllo residui fuori dalle stringhe
    cleaned = re.sub(r'[\x00-\x1f\x7f](?=[^"]*(?:"[^"]*"[^"]*)*$)', '', cleaned)

    # 6. Ultima prova
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise ValueError(f"Impossibile parsare la stringa JSON dopo la pulizia: {e}")

def safe_json_loads(text: str):
    try:
        return clean_and_parse_json(text), None
    except Exception as e:
        return None, str(e)

def validate_schema(obj: dict):
    """Validazione minimale del tuo schema (campi + confidence 0..100)."""
    required = ["answer", "decision", "preferred_brand", "confidence", "confidence_rationale", "sources"]
    missing = [k for k in required if k not in obj]
    if missing:
        return False, f"Missing keys: {missing}"
    if not isinstance(obj["confidence"], int) or not (0 <= obj["confidence"] <= 100):
        return False, "confidence must be int in [0..100]"
    if not isinstance(obj["sources"], list):
        return False, "sources must be a list"
    return True, None

In [27]:
answers = []
for i, item in enumerate(prompts[3:7]):
    user_prompt = item["prompt"]
    tone = item.get("tone")
    lang = item.get("language")

    for model in MODELS:
        t0 = time.time()
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
            )
            content = resp.choices[0].message.content

            parsed, parse_err = safe_json_loads(content)
            schema_ok, schema_err = (False, None)
            if parsed is not None and isinstance(parsed, dict):
                schema_ok, schema_err = validate_schema(parsed)

            record = {
                "timestamp": datetime.now(),
                "prompt_index": i,
                "tone": tone,
                "language": lang,
                "model": model,
                "user_prompt": user_prompt,

                "raw_response": content,
                "parsed_json": parsed,

                "json_parse_error": parse_err,
                "schema_ok": schema_ok,
                "schema_error": schema_err,

                "latency_s": round(time.time() - t0, 3),
                "usage": getattr(resp, "usage", None),
            }

        except Exception as e:
            record = {
                "timestamp": datetime.now(),
                "prompt_index": i,
                "tone": tone,
                "language": lang,
                "model": model,
                "user_prompt": user_prompt,
                "error": str(e),
                "latency_s": round(time.time() - t0, 3),
            }
        answers.append(record)

In [33]:
for iteration in answers:
    preferred_brand = iteration.get("parsed_json", {}).get("preferred_brand", "")
    confidence = iteration.get("parsed_json", {}).get("confidence", "")
    model = iteration.get("model", "")
    print(f"Model: {model} | Preferred Brand: {preferred_brand} | Confidence: {confidence}")

Model: claude_3_5_haiku | Preferred Brand: Frontline | Confidence: 85
Model: gpt-4o-mini | Preferred Brand: Frontline | Confidence: 90
Model: claude_3_5_haiku | Preferred Brand: Bravecto | Confidence: 85
Model: gpt-4o-mini | Preferred Brand: Bravecto | Confidence: 90
Model: claude_3_5_haiku | Preferred Brand: Frontline Plus | Confidence: 85
Model: gpt-4o-mini | Preferred Brand: Frontline | Confidence: 90
Model: claude_3_5_haiku | Preferred Brand: Nexgard | Confidence: 85
Model: gpt-4o-mini | Preferred Brand: NexGard | Confidence: 90
